# masktools -- two ways to make a pixel mask

The package has exactly two functions, and this notebook is the whole of it.

| function | what it does |
| --- | --- |
| `mask_poly_expand(mask, factor)` | grow or shrink every region of a mask |
| `mask_manual(image, mask=...)` | trace regions with the mouse and get a mask back |

Both speak the same data type: a 2-D boolean array in which `True` means "this
pixel is masked".  There is nothing else to learn.

We work on a real frame -- a 512x512 cutout of an EFOSC2 exposure that ships with
the package -- which already carries an example mask, so both functions have
something to work on straight away.


In [ ]:
%matplotlib widget
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from astropy.io import fits
from astropy.visualization import simple_norm

from masktools import mask_manual, mask_poly_expand

# Flip to False to skip the mouse step, e.g. when running this notebook as a
# script; the drawing cell then reuses the shipped mask unchanged.
DRAW_BY_HAND = True

# Locate the example frame wherever the notebook was opened from.
candidate = [Path.cwd(), *Path.cwd().parents]
where = next((p for p in candidate if (p / "data" / "example_cutout.fits.gz").exists()), None)
assert where is not None, "run this from the masktools repo, or set DATA by hand"
DATA = where / "data" / "example_cutout.fits.gz"

with fits.open(DATA) as hdul:
    image = np.asarray(hdul[0].data, float)       # the science pixels
    header = hdul[0].header                       # its WCS, for later
    mask = np.asarray(hdul[1].data).astype(bool)  # the shipped example mask

sky = np.median(image)
print(DATA)
print(f"{image.shape[0]}x{image.shape[1]} pixels, sky level {sky:.1f}, peak {image.max():.0f}")
print(f"shipped mask: {mask.sum()} pixels")


## 0. The frame, and the mask it ships with

`%matplotlib widget` gives interactive inline plots (it needs `ipympl`).  The
drawing window in section 2 does *not* depend on it -- it opens its own window
whatever backend the notebook uses.


In [ ]:
def show(mask, title, zoom=None, compare=None):
    """One panel: the frame, *mask* outlined in red, *compare* in cyan."""
    norm = simple_norm(image, stretch="asinh", min_percent=16, max_percent=99.9)
    fig, ax = plt.subplots(figsize=(6, 6))
    ax.imshow(image, cmap="Greys_r", norm=norm, origin="lower")
    if compare is not None:
        ax.contour(compare, levels=[0.5], colors="cyan", linewidths=1.0, origin="lower")
    ax.contour(mask, levels=[0.5], colors="red", linewidths=1.4, origin="lower")
    ax.set_title(f"{title} -- {mask.sum()} pixels")
    if zoom is not None:
        r0, r1, c0, c1 = zoom
        ax.set_xlim(c0, c1)
        ax.set_ylim(r0, r1)
    return fig, ax


rows, cols = np.nonzero(mask)
ZOOM = (rows.min() - 25, rows.max() + 25, cols.min() - 25, cols.max() + 25)
show(mask, "the shipped example mask", ZOOM);


Those 582 masked pixels are the interesting part: **every one of them is exactly
zero** in the science array, while the frame's sky sits at 494.  They form a
solid 24x25 block centred 1.3 pixels from the photocentre of the star you can see
there.  In other words the reduction pipeline flagged the star's core, and the
mask marks the dead pixels -- not the star itself.

That distinction is the reason to read section 1: a mask on the core alone throws
away nothing of the star's light, which is usually *not* what you want when the
point is to exclude this star from a measurement.


## 1. Expanding a mask

`mask_poly_expand(mask, factor)` resizes every masked region and leaves the rest
of the frame alone.  *factor* is a **linear** factor, like a photocopy
enlargement, so the pixel count goes as `factor**2`:

- `factor=2` -> twice as wide *and* twice as tall -> **four times** the pixels
- `factor=0.5` -> half as wide -> a quarter of the pixels
- `factor=1` -> the same mask back, bit for bit

On a plain 10x10 square, where the arithmetic is easy to check:


In [ ]:
square = np.zeros((40, 40), bool)
square[10:20, 10:20] = True          # 100 pixels

for factor in (0.5, 1.0, 1.5, 2.0, 3.0):
    big = mask_poly_expand(square, factor)
    print(f"factor {factor:>4}: {big.sum():4d} pixels   (100 x {factor}^2 = {100 * factor**2:.0f})")


Each region is resized about **its own** centre, so separate regions keep their
separation -- nothing slides toward the middle of the frame.

Now the same thing on the real star.  Cyan is the original 582-pixel core, red is
the expanded mask:


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5.6), sharex=True, sharey=True)
norm = simple_norm(image, stretch="asinh", min_percent=16, max_percent=99.9)

grown = {factor: mask_poly_expand(mask, factor) for factor in (1.5, 2.0, 3.0)}
rows_g, cols_g = np.nonzero(grown[3.0])          # the widest of the three
PAD = 4
limits = (rows_g.min() - PAD, rows_g.max() + PAD, cols_g.min() - PAD, cols_g.max() + PAD)

for ax, factor in zip(axes, (1.5, 2.0, 3.0)):
    ax.imshow(image, cmap="Greys_r", norm=norm, origin="lower")
    ax.contour(mask, levels=[0.5], colors="cyan", linewidths=1.0, origin="lower")
    ax.contour(grown[factor], levels=[0.5], colors="red", linewidths=1.4, origin="lower")
    ax.set_title(f"factor {factor}: {grown[factor].sum()} pixels")
    ax.set_xlim(limits[2], limits[3])
    ax.set_ylim(limits[0], limits[1])

axes[0].set_ylabel("row (pixel)")
axes[2].set_xlabel("column (pixel)")
fig.tight_layout()


### Was that worth it?  Measure what the mask now covers

"Big enough" is not a property of the mask, it is a property of the star.  So ask
the frame: what fraction of the star's light falls inside the mask?  Below,
`bright` is every pixel in the star's neighbourhood standing more than 3 sigma
above sky -- the light you would actually want excluded.


In [ ]:
sigma = 1.4826 * np.median(np.abs(image - sky))
window = np.zeros_like(mask)
window[ZOOM[0]:ZOOM[1], ZOOM[2]:ZOOM[3]] = True          # this star only
bright = ((image - sky) > 3 * sigma) & window

excess = np.clip(image - sky, 0, None)
total_flux = excess[bright].sum()
print(f"{bright.sum()} bright pixels around the star, {total_flux:.0f} ADU of excess flux")
print()

print(f"{'factor':>7} | {'pixels':>7} | {'bright pixels covered':>22} | {'flux covered':>13}")
print("-" * 56)
for factor in (1.0, 1.5, 2.0, 2.5, 3.0):
    big = mask_poly_expand(mask, factor)
    n_cov = (bright & big).sum()
    f_cov = excess[bright & big].sum() / total_flux
    print(f"{factor:>7} | {big.sum():>7} | {n_cov:>10} of {bright.sum():<10} | {100 * f_cov:>12.1f}%")


The flagged core on its own covers **none** of the star's light.  A factor of 2
already takes 96% of it, and a factor of 3 takes all of it.  That is the practical
decision *mask_poly_expand* is there to serve: pick the factor by what it covers,
not by how the outline looks.

One thing to watch: the mask cannot grow past the edges of the array.  If a
region reaches the border you get a warning and the overflowing part is simply
dropped -- pad the frame with `numpy.pad` first if you need that room.


## 2. Drawing a mask by hand

`mask_manual(image, mask=...)` opens a window on the frame and lets you trace
regions with the mouse.  Run the cell below and the notebook **waits** until you
are finished, then hands you the mask.

| key | action |
| --- | --- |
| left click | place a vertex |
| click the first vertex again | close the polygon (needs 3+ vertices) |
| drag a vertex | move it |
| `shift` + drag | move a whole polygon |
| right click a vertex | delete it |
| `n` | finish this region, start the next |
| `d` | delete the last finished region |
| `q` | **done** -- return the mask |
| `h` | print this list again |

Pass the mask you already have and its regions come in as editable outlines, so
you are refining rather than starting over.  Closing the window without drawing
anything loses nothing: you get the mask you passed in.


In [ ]:
if DRAW_BY_HAND:
    drawn = mask_manual(image, mask=mask)
else:
    drawn = mask.copy()   # automated run: nothing to trace

print(f"drawn mask: {drawn.sum()} pixels (started from {mask.sum()})")
print("nothing is written to disk yet -- the last cell saves it; to save from the window itself, pass savepath=... and press s")


In [ ]:
show(drawn, "after drawing by hand", ZOOM, compare=mask);


## 3. The two together

They compose, because they are just arrays in and arrays out.  A typical
sequence: trace a region around something the automatic mask missed, then pad it
out to be safe.


In [ ]:
final = mask_poly_expand(drawn, 1.2)
print(f"hand-drawn {drawn.sum()} pixels -> expanded {final.sum()} pixels")
show(final, "hand-drawn, then expanded by 1.2", ZOOM, compare=drawn);


## 4. Saving it

Give the FITS writer the frame's own header and the WCS travels with the mask, so
it stays lined up with the image:


In [ ]:
out = where / "examples" / "output" / "my_mask.fits"
out.parent.mkdir(exist_ok=True)          # scratch directory, not tracked by git
fits.PrimaryHDU(drawn.astype(np.uint8), header=header).writeto(out, overwrite=True)

back = np.asarray(fits.getdata(out)).astype(bool)
print(f"wrote {out.name}: {back.sum()} pixels, round trip identical: {np.array_equal(back, drawn)}")

from astropy.wcs import WCS
r0, c0 = (int(v[0]) for v in np.nonzero(drawn))
print(f"pixel (row {r0}, col {c0}) -> {WCS(header).pixel_to_world(c0, r0)}")


## That is the whole package

```python
mask_poly_expand(mask, factor)   # resize every region, linearly
mask_manual(image, mask)         # trace regions, get a boolean mask back
```

Both take and return plain 2-D boolean arrays, so anything that accepts a numpy
mask -- photutils, your own summation, a later reduction -- accepts their output
unchanged.
